# ERA5 to PRISM Downscaling (Georgia)
Multi-variable regional downscaling with persistence, linear, CNN, and ConvLSTM models.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets').exists() and (ROOT.parent / 'datasets').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')


## Problem
- ERA5 is coarse reanalysis data.
- PRISM is a higher-resolution temperature target.
- The mapping is `ERA5(t-k+1 ... t) -> PRISM(t)`.


## Modeling path
Persistence and linear baselines are included first. CNN is the spatial baseline. ConvLSTM is the temporal model.

The next useful step here was stabilizing temporal training, not adding more model complexity.

## Multi-variable inputs
Temperature-only input was too limited. ERA5 now includes temperature, 10m wind (u/v), and surface pressure.

In [ ]:
import json
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from datasets.prism_dataset import ERA5_PRISM_Dataset

ERA5_PATH = ROOT / 'data_raw/era5_georgia_temp.nc'
PRISM_DIR = ROOT / 'data_raw/prism'
RESULTS_DIR = ROOT / 'results/evaluation'

if ERA5_PATH.exists() and PRISM_DIR.exists():
    ds_era5 = xr.open_dataset(ERA5_PATH)
    print('ERA5 variables:', list(ds_era5.data_vars))
    dataset = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=3, verbose=False)
    x, y = dataset[0]
    print('Usable samples:', len(dataset))
    print('Input shape [T, C, H, W]:', tuple(x.shape))
    print('Target shape [C, H, W]:', tuple(y.shape))
else:
    print('Data is missing. Run:')
    print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
    print('python data_pipeline/download_prism.py --start-date 20230101 --days 30 --variable tmean')


## Results

In [ ]:
def latest_png(folder: Path):
    files = sorted(folder.glob('comparison_*.png'))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

cnn_img = latest_png(RESULTS_DIR / 'cnn')
convlstm_img = latest_png(RESULTS_DIR / 'convlstm')
summary_csv = RESULTS_DIR / 'baselines_summary.csv'
comparison_img = RESULTS_DIR / 'model_comparison.png'
cnn_metrics = RESULTS_DIR / 'cnn' / 'metrics.json'
convlstm_metrics = RESULTS_DIR / 'convlstm' / 'metrics.json'

if cnn_img:
    display(Markdown('### CNN output'))
    display(Image(filename=str(cnn_img)))
if convlstm_img:
    display(Markdown('### ConvLSTM output'))
    display(Image(filename=str(convlstm_img)))
if comparison_img.exists():
    display(Markdown('### Model comparison'))
    display(Image(filename=str(comparison_img)))

rows = []
for name, path in [('cnn', cnn_metrics), ('convlstm', convlstm_metrics)]:
    if path.exists():
        m = json.loads(path.read_text())
        rows.append({'model': name, 'rmse': m.get('rmse'), 'mae': m.get('mae'), 'bias': m.get('bias')})
if rows:
    display(Markdown('### CNN vs ConvLSTM metrics'))
    display(pd.DataFrame(rows))

if summary_csv.exists():
    display(Markdown('### Full model table'))
    display(pd.read_csv(summary_csv))

missing = (cnn_img is None) or (convlstm_img is None) or (not summary_csv.exists()) or (not comparison_img.exists())
if missing:
    print('Run:')
    print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
    print('python data_pipeline/download_prism.py --start-date 20230101 --days 30 --variable tmean')
    print('python training/train_downscaler.py --model cnn --history-length 3 --epochs 25 --learning-rate 1e-3 --split-seed 42')
    print('python training/train_downscaler.py --model convlstm --history-length 3 --epochs 40 --learning-rate 3e-4 --split-seed 42')
    print('python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 3 --num-samples 8 --split-seed 42')


## Interpretation
- ConvLSTM is now much more stable than earlier runs and beats CNN in this setup.
- Linear baseline is still strongest on this data slice.
- Stronger models need enough data and careful training settings.
- Multi-variable input makes this pipeline more realistic for temporal downscaling.


## Limitations and next step
Current limitations are still data volume and region coverage. The direction remains temporal training with longer windows and broader data support, not extra model complexity.